In [ ]:
"""
Netherlands 3D BAG -> Monocular Per-Pixel Building-Relative Height Regressor
End-to-end pipeline: Tiling, 3D BAG ingestion, leakage-free spatial splitting,
U-Net (ResNet-18) training, evaluation, INT8 quantization & deployment.
"""

import os
import sys
import math
import json
import shutil
import time
import hashlib
import threading
import collections
import concurrent.futures as cf
import csv

# PyTorch CUDA allocator tuning
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import requests
import cv2
import fiona
try:
    from fiona.model import Geometry as _FionaGeometry
except Exception:
    _FionaGeometry = None

from pyproj import Transformer
from shapely.geometry import shape, box, mapping
from shapely.ops import transform as shp_transform
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18
try:
    from torchvision.models import ResNet18_Weights
    _IMAGENET1K = ResNet18_Weights.DEFAULT
except Exception:
    _IMAGENET1K = "IMAGENET1K_V1"

# Environment auto-detection
IN_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ
DEFAULT_WORK_DIR = "/content/drive/MyDrive/bag_height" if IN_COLAB else os.path.abspath("./bag_height_data")

CONFIG = dict(
    work_dir=os.environ.get("BAG_WORK_DIR", DEFAULT_WORK_DIR),
    mount_drive=IN_COLAB,
    resume=True,
    keep_tile_cache=False,
    patch_batch=32,
    build_limit=None,
    restrict_to_city=True,
    city_name="amsterdam_metro_extended",
    city_bbox=(4.60, 52.04, 5.28, 52.44),
    bag_source="tiles",
    bag_tile_index_url="https://data.3dbag.nl/latest/tile_index.fgb",
    bag_gpkg_path=None,
    verify_sha256=True,
    keep_bag_tiles=True,
    imagery_provider="esri",
    zoom=16,
    patch_tiles=3,
    model_imgsz=640,
    max_patches=1000,
    min_buildings=5,
    region_patches=8,
    seed=42,
    block_patches=8,
    split_chunk_blocks=40,
    buffer_blocks=0,
    train_frac=0.80,
    val_frac=0.10,
    test_frac=0.10,
    min_val_test=10,
    download_workers=16,
    jpeg_quality=92,
    blank_std_threshold=8.0,
    height_min=0.0,
    height_max=120.0,
    epochs=80,
    micro_batch=4,
    effective_batch=16,
    lr=1e-4,
    weight_decay=1e-4,
    huber_beta=1.0,
    lambda_bg=0.05,
    num_workers=2,
    amp=True,
    device=0,
    viz_n=10,
    quantization_enabled=True,
    quant_bits=8,
    quant_calib_samples=24,
    cpu_inference_enabled=True,
    onnx_opset=13,
    quant_per_channel=True,
    quant_activation_type="uint8",
    quant_weight_type="int8",
    onnx_imgsz=None,
    onnx_compare_samples=16,
    onnx_bench_runs=10,
    onnx_bench_warmup=2,
    quant_exclude_output_nodes=True,
)

PROFILE = os.environ.get("BAG_PROFILE", "custom")
_PRESETS = {
    "quick": dict(epochs=3, max_patches=16, micro_batch=2, effective_batch=8),
    "full": dict(epochs=60, max_patches=None),
}
CONFIG.update(_PRESETS.get(PROFILE, {}))

for _k in list(CONFIG):
    _env = os.environ.get("BAG_" + _k.upper())
    if _env is not None:
        _cur = CONFIG[_k]
        if _k in ("max_patches", "build_limit", "onnx_imgsz"):
            CONFIG[_k] = None if _env.strip().lower() in ("", "none", "all", "-1") else int(_env)
        elif _k == "bag_gpkg_path":
            CONFIG[_k] = None if _env.strip().lower() in ("", "none", "all", "-1") else _env
        elif _k in ("city_bbox", "aoi_bbox"):
            CONFIG[_k] = tuple(float(v) for v in _env.split(","))
        elif _k == "device":
            CONFIG[_k] = int(_env) if _env.isdigit() else _env
        elif isinstance(_cur, bool):
            CONFIG[_k] = _env.lower() in ("1", "true", "yes", "on")
        elif isinstance(_cur, int):
            CONFIG[_k] = int(_env)
        elif isinstance(_cur, float):
            CONFIG[_k] = float(_env)
        else:
            CONFIG[_k] = _env

for k, v in CONFIG.items():
    globals()[k] = v

assert restrict_to_city, "Pipeline scoped to ONE dense urban area."
assert len(city_bbox) == 4 and city_bbox[0] < city_bbox[2] and city_bbox[1] < city_bbox[3]
SCOPE = city_name
aoi_name = SCOPE
aoi_bbox = tuple(float(v) for v in city_bbox)

if not torch.cuda.is_available():
    device = "cpu"

assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-9
assert effective_batch % micro_batch == 0
assert height_max > height_min
assert model_imgsz % 32 == 0

AOI_LAT_C = 0.5 * (aoi_bbox[1] + aoi_bbox[3])
GSD_M_PER_PX = 156543.03 * math.cos(math.radians(AOI_LAT_C)) / (2 ** zoom)
PATCH_PX = patch_tiles * 256
STRIDE_PX = patch_tiles * 256
PATCH_GROUND_M = PATCH_PX * GSD_M_PER_PX

SHARED_DATA_DIR = os.path.join(work_dir, "data")
RUNS_DIR = os.path.join(work_dir, "runs", SCOPE)
DATA_DIR = os.path.join(RUNS_DIR, "data")
CACHE_DIR = os.path.join(RUNS_DIR, "tile_cache")
BAG_TILES_DIR = os.path.join(DATA_DIR, "bag_tiles")
IMG_DIR = os.path.join(RUNS_DIR, "images")
TGT_DIR = os.path.join(RUNS_DIR, "targets")
SPLITS_DIR = os.path.join(work_dir, "splits", SCOPE)
LOGS_DIR = os.path.join(RUNS_DIR, "logs")
CHECKPOINT_DIR = os.path.join(RUNS_DIR, "checkpoints")
EVAL_DIR = os.path.join(RUNS_DIR, "eval")
VIZ_DIR = os.path.join(RUNS_DIR, "viz")
PRED_DIR = os.path.join(RUNS_DIR, "predictions")
ONNX_DIR = os.path.join(RUNS_DIR, "onnx")
BL_DIR = os.path.join(RUNS_DIR, "building_level_eval")

for d in (work_dir, SHARED_DATA_DIR, RUNS_DIR, DATA_DIR, CACHE_DIR, BAG_TILES_DIR,
          SPLITS_DIR, LOGS_DIR, CHECKPOINT_DIR, EVAL_DIR, VIZ_DIR, PRED_DIR,
          ONNX_DIR, BL_DIR):
    os.makedirs(d, exist_ok=True)

STATE_PATH = os.path.join(work_dir, f"pipeline_state_{SCOPE}.json")
MANIFEST_PATH = os.path.join(work_dir, f"download_manifest_{SCOPE}.jsonl")
TILE_INDEX_PATH = os.path.join(SHARED_DATA_DIR, "tile_index.fgb")
HIST_TAG = f"z{zoom}_p{patch_tiles}"
FGB_PATH = os.path.join(DATA_DIR, f"{aoi_name}_buildings.fgb")
HIST_PATH = os.path.join(DATA_DIR, f"{aoi_name}_hist_{HIST_TAG}.npz")
SPLIT_META = os.path.join(SPLITS_DIR, "split_meta.json")
SPLIT_FILES = {n: os.path.join(SPLITS_DIR, f"{n}.txt") for n in ("train", "val", "test")}
LAST_PT = os.path.join(CHECKPOINT_DIR, "last.pt")
BEST_PT = os.path.join(CHECKPOINT_DIR, "best.pt")
METRICS_PATH = os.path.join(EVAL_DIR, "metrics.json")
RUN_SUMMARY_PATH = os.path.join(RUNS_DIR, "run_summary.json")
ONNX_FP32 = os.path.join(ONNX_DIR, "best_fp32.onnx")
ONNX_INT8 = os.path.join(ONNX_DIR, "best_int8.onnx")
QUANT_METRICS_PATH = os.path.join(ONNX_DIR, "quantization_metrics.json")
CALIB_JSON = os.path.join(ONNX_DIR, "calibration_samples.json")
QUANT_META = os.path.join(ONNX_DIR, "quant_meta.json")
BL_CSV = os.path.join(BL_DIR, "building_height_instance_report.csv")

if mount_drive and IN_COLAB:
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")
    except Exception as e:
        print(f"[DRIVE] Mount skipped ({e}); using work_dir locally.")

LOG_PATH = os.path.join(LOGS_DIR, "pipeline.log")

def log(msg):
    line = f"{time.strftime('%Y-%m-%d %H:%M:%S')} {msg}"
    print(msg)
    try:
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(line + "\n")
    except Exception:
        pass

STAGES = ["dataset_download", "index_build", "split_generation",
          "patch_generation", "training", "evaluation", "visualization",
          "onnx_export", "quantization", "cpu_inference"]

def load_state():
    if os.path.exists(STATE_PATH):
        try:
            return json.load(open(STATE_PATH, encoding="utf-8"))
        except Exception:
            pass
    return {s: "pending" for s in STAGES}

def save_state(state):
    tmp = STATE_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)
    os.replace(tmp, STATE_PATH)

STATE = load_state()

def set_stage(name, status, **extra):
    STATE[name] = status
    STATE.setdefault("meta", {}).update(extra)
    STATE["updated"] = time.strftime("%Y-%m-%d %H:%M:%S")
    save_state(STATE)
    log(f"[STATE] {name} = {status}")

def record_batch_state(failed=None, good=None):
    mem = STATE.setdefault("mem", {})
    changed = False
    if failed is not None and mem.get("failed_micro_batch") != int(failed):
        mem["failed_micro_batch"] = int(failed)
        changed = True
    if good is not None and mem.get("last_good_micro_batch") != int(good):
        mem["last_good_micro_batch"] = int(good)
        changed = True
    if changed:
        mem["updated"] = time.strftime("%Y-%m-%d %H:%M:%S")
        save_state(STATE)
        log(f"[MEM] recorded batch history: {json.dumps(mem)}")

MANIFEST_FIELDS = ["patch_id", "split", "source", "status", "image_path",
                   "target_path", "file_size", "n_polygons", "n_valid_buildings",
                   "n_valid_px", "validated", "error"]

def load_manifest():
    rows = {}
    if not os.path.exists(MANIFEST_PATH):
        return rows
    try:
        with open(MANIFEST_PATH, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    r = json.loads(line)
                except Exception:
                    continue
                if r.get("patch_id"):
                    rows[r["patch_id"]] = r
    except Exception as e:
        log(f"[WARN] manifest unreadable ({e}); starting fresh")
    return rows

def append_manifest(records):
    if not records:
        return
    with open(MANIFEST_PATH, "a", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps({k: r.get(k, "") for k in MANIFEST_FIELDS}) + "\n")

IMAGERY_PROVIDERS = {
    "esri": {
        "url": ("https://server.arcgisonline.com/ArcGIS/rest/services/"
                "World_Imagery/MapServer/tile/{z}/{y}/{x}"),
        "referer": "https://www.arcgis.com/",
    },
    "eox_s2cloudless": {
        "url": ("https://tiles.maps.eox.at/wmts/1.0.0/"
                "s2cloudless-2020_3857/default/g/{z}/{y}/{x}.jpg"),
        "referer": "https://s2maps.eu/",
    },
}
_provider = IMAGERY_PROVIDERS[imagery_provider]
TILE_URL = _provider["url"]
TILE_REFERER = _provider.get("referer", "")

def lonlat_to_tile_f(lon, lat, z):
    n = 2.0 ** z
    x = (lon + 180.0) / 360.0 * n
    lat_rad = math.radians(lat)
    y = (1.0 - math.asinh(math.tan(lat_rad)) / math.pi) / 2.0 * n
    return x, y

def tile_to_lonlat(x, y, z):
    n = 2.0 ** z
    lon = x / n * 360.0 - 180.0
    lat = math.degrees(math.atan(math.sinh(math.pi * (1.0 - 2.0 * y / n))))
    return lon, lat

def patch_bbox_lonlat(px, py):
    ox, oy = px * patch_tiles, py * patch_tiles
    lon0, lat0 = tile_to_lonlat(ox, oy, zoom)
    lon1, lat1 = tile_to_lonlat(ox + patch_tiles, oy + patch_tiles, zoom)
    return (lon0, lat1, lon1, lat0)

_TF_4326_3857 = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
WEB_MERCATOR_MAX = 20037508.342789244

def lonlat_to_webmercator(lon, lat):
    return _TF_4326_3857.transform(lon, lat)

def webmercator_resolution(z=zoom):
    world_px = 256.0 * (2 ** z)
    return 2.0 * WEB_MERCATOR_MAX / world_px

class PatchProjector:
    def __init__(self, patch_x, patch_y, z=zoom):
        self.ox = patch_x * patch_tiles
        self.oy = patch_y * patch_tiles
        self.res = webmercator_resolution(z)
        self.origin_x = -WEB_MERCATOR_MAX + self.ox * 256.0 * self.res
        self.origin_y = WEB_MERCATOR_MAX - self.oy * 256.0 * self.res

    def to_pixels(self, geom_lonlat):
        g = shp_transform(lambda x, y, z=None: _TF_4326_3857.transform(x, y), geom_lonlat)
        xy = np.asarray(g.exterior.coords, dtype=np.float64)
        px = (xy[:, 0] - self.origin_x) / self.res
        py = (self.origin_y - xy[:, 1]) / self.res
        np.clip(np.round(px), 0, PATCH_PX, out=px)
        np.clip(np.round(py), 0, PATCH_PX, out=py)
        return np.stack([px, py], axis=1).astype(np.int32)

_tls = threading.local()

def _session():
    s = getattr(_tls, "s", None)
    if s is None:
        s = requests.Session()
        headers = {
            "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122 Safari/537.36"),
            "Accept": "image/avif,image/webp,image/jpeg,image/png,*/*",
        }
        if TILE_REFERER:
            headers["Referer"] = TILE_REFERER
        s.headers.update(headers)
        _tls.s = s
    return s

def fetch_tile(z, x, y, retries=4):
    fp = os.path.join(CACHE_DIR, f"{z}_{x}_{y}.jpg")
    if os.path.exists(fp) and os.path.getsize(fp) > 0:
        return fp
    url = TILE_URL.format(z=z, y=y, x=x)
    for attempt in range(retries):
        try:
            r = _session().get(url, timeout=30)
            if r.status_code == 200 and r.content and len(r.content) > 100:
                with open(fp, "wb") as f:
                    f.write(r.content)
                return fp
        except Exception:
            pass
        time.sleep(0.5 * (attempt + 1))
    return None

def patch_tile_list(px, py):
    ox, oy = px * patch_tiles, py * patch_tiles
    return [(zoom, ox + dx, oy + dy) for dy in range(patch_tiles) for dx in range(patch_tiles)]

def stitch_patch(px, py):
    ox, oy = px * patch_tiles, py * patch_tiles
    canvas = Image.new("RGB", (PATCH_PX, PATCH_PX), (0, 0, 0))
    ok = 0
    for dy in range(patch_tiles):
        for dx in range(patch_tiles):
            fp = os.path.join(CACHE_DIR, f"{zoom}_{ox + dx}_{oy + dy}.jpg")
            if not os.path.exists(fp):
                continue
            canvas.paste(Image.open(fp).convert("RGB"), (dx * 256, dy * 256))
            ok += 1
    if ok < patch_tiles * patch_tiles:
        return None
    return canvas

def drop_tiles(tiles):
    if keep_tile_cache:
        return
    for z, tx, ty in tiles:
        fp = os.path.join(CACHE_DIR, f"{z}_{tx}_{ty}.jpg")
        try:
            if os.path.exists(fp):
                os.remove(fp)
        except OSError:
            pass

def valid_image(path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return False
    try:
        with Image.open(path) as im:
            im.verify()
        return True
    except Exception:
        return False

def valid_target(path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return False
    try:
        with np.load(path) as d:
            if "height" not in d or "mask" not in d:
                return False
            h, m = d["height"], d["mask"]
            return (h.shape == (PATCH_PX, PATCH_PX) and h.dtype == np.float32
                    and m.shape == (PATCH_PX, PATCH_PX) and m.dtype == np.uint8)
    except Exception:
        return False

def _part1by1(n):
    n &= 0x0000FFFF
    n = (n | (n << 8)) & 0x00FF00FF
    n = (n | (n << 4)) & 0x0F0F0F0F
    n = (n | (n << 2)) & 0x33333333
    n = (n | (n << 1)) & 0x55555555
    return n

def morton(x, y):
    return (_part1by1(x) << 1) | _part1by1(y)

def stratified_sample(px, py, counts, n, rng):
    if n <= 0 or len(px) == 0:
        return np.zeros(0, dtype=np.int64)
    region = (px // region_patches).astype(np.int64) * 100000 + (py // region_patches)
    order = np.argsort(region, kind="stable")
    region_s, cnt_s = region[order], counts[order]
    bounds = np.searchsorted(region_s, np.unique(region_s), side="left").tolist()
    bounds.append(len(region_s))
    n_regions = len(bounds) - 1
    per_region = max(1, int(math.ceil(n / max(1, n_regions))))
    picked = []
    for i in range(n_regions):
        lo, hi = bounds[i], bounds[i + 1]
        k = min(per_region, hi - lo)
        local = np.arange(lo, hi)
        w = cnt_s[lo:hi].astype(np.float64)
        w = w / w.sum() if w.sum() > 0 else None
        sel = rng.choice(local, size=k, replace=False, p=w)
        picked.append(sel)
    sel = np.concatenate(picked) if picked else np.zeros(0, dtype=np.int64)
    if len(sel) > n:
        sel = rng.choice(sel, size=n, replace=False)
    return order[sel]

def _download_url(url, dst, sha256=None, retries=5):
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        if not (verify_sha256 and sha256):
            return False
        h = hashlib.sha256()
        with open(dst, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
        if h.hexdigest() == sha256:
            return False
        os.remove(dst)
    tmp = dst + f".tmp{os.getpid()}"
    for attempt in range(retries):
        try:
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        if chunk:
                            f.write(chunk)
            if sha256 and verify_sha256:
                h = hashlib.sha256()
                with open(tmp, "rb") as f:
                    for chunk in iter(lambda: f.read(1 << 20), b""):
                        h.update(chunk)
                if h.hexdigest() != sha256:
                    os.remove(tmp)
                    raise RuntimeError(f"sha256 mismatch for {url}")
            os.replace(tmp, dst)
            return True
        except Exception:
            try:
                if os.path.exists(tmp):
                    os.remove(tmp)
            except OSError:
                pass
            time.sleep(1.0 * (attempt + 1))
    raise RuntimeError(f"download failed after {retries} attempts: {url}")

def aoi_rd_bbox():
    tr = Transformer.from_crs("EPSG:4326", "EPSG:28992", always_xy=True)
    x0, y0 = tr.transform(aoi_bbox[0], aoi_bbox[1])
    x1, y1 = tr.transform(aoi_bbox[2], aoi_bbox[3])
    return (min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1))

def select_bag_tiles():
    bbox_rd = aoi_rd_bbox()
    with fiona.open(TILE_INDEX_PATH) as src:
        log(f"[CHECK] 3D BAG tile index: {len(src):,} tiles | crs={src.crs}")
        feats = list(src.filter(bbox=bbox_rd))
    tiles = []
    for f in feats:
        p = dict(f["properties"])
        tiles.append((p["tile_id"], p["gpkg_download"], p.get("gpkg_sha256")))
    tiles.sort(key=lambda t: t[0])
    return tiles

if bag_source == "local":
    assert bag_gpkg_path, "bag_source='local' requires bag_gpkg_path"
    assert os.path.exists(bag_gpkg_path), f"missing local 3D BAG file: {bag_gpkg_path}"
    set_stage("dataset_download", "complete", source="local", path=bag_gpkg_path)
else:
    if resume and os.path.exists(TILE_INDEX_PATH) and os.path.getsize(TILE_INDEX_PATH) > 0:
        log("[CHECK] tile index cached -> SKIP download")
    else:
        log(f"[DOWNLOAD] 3D BAG tile index: {bag_tile_index_url}")
        _download_url(bag_tile_index_url, TILE_INDEX_PATH, sha256=None)
    bag_tiles = select_bag_tiles()
    log(f"[CHECK] {len(bag_tiles)} tiles intersect city_bbox")

    def _tile_path(tile_id):
        return os.path.join(BAG_TILES_DIR, tile_id.replace("/", "_") + ".gpkg.gz")

    stats = {"downloaded": 0, "verified": 0, "failed": 0}
    with cf.ThreadPoolExecutor(max_workers=download_workers) as ex:
        futs = {ex.submit(_download_url, url, _tile_path(tid), sha): tid
                for tid, url, sha in bag_tiles}
        for fut in cf.as_completed(futs):
            tid = futs[fut]
            try:
                if fut.result():
                    stats["downloaded"] += 1
                else:
                    stats["verified"] += 1
            except Exception as e:
                stats["failed"] += 1
                log(f"[FAIL] 3D BAG tile {tid}: {e}")
    set_stage("dataset_download",
              "complete" if stats["failed"] == 0 else "in_progress",
              n_tiles=len(bag_tiles), new=stats["downloaded"], failed=stats["failed"])

ROOF_ATTR_CANDIDATES = ["h_dak_70p", "h_roof_70p", "b3_h_dak_70p", "b3_h_roof_70p", "b3_h_70p", "h_70p"]
GROUND_ATTR_CANDIDATES = ["h_maaiveld", "h_ground", "b3_h_maaiveld", "b3_h_ground"]
FOOTPRINT_LAYER_CANDIDATES = ["pand", "gebouw", "building", "bag_pand"]
ROOFPART_LAYER_CANDIDATES = ["lod22_2d", "lod13_2d", "lod12_2d", "pand_dak"]
ROOFPART_ROOF_CANDIDATES = ["b3_h_70p", "h_dak_70p", "h_roof_70p", "b3_h_dak_70p", "h_70p"]
JOIN_KEY_CANDIDATES = ["identificatie", "bag_id", "pand_identificatie"]
import gzip

def _first_present(candidates, available):
    for c in candidates:
        if c in available:
            return c
    return None

def resolve_height_schema(src_path):
    layers = fiona.listlayers(src_path)
    fp_layer = _first_present(FOOTPRINT_LAYER_CANDIDATES, layers)
    if fp_layer is None:
        raise SystemExit(f"no footprint layer in {src_path}; layers={layers}")
    with fiona.open(src_path, layer=fp_layer) as lyr:
        props = dict(lyr.schema["properties"])
    ground_attr = _first_present(GROUND_ATTR_CANDIDATES, props)
    if ground_attr is None:
        raise SystemExit(f"ground elevation not found on '{fp_layer}'")
    roof_attr = _first_present(ROOF_ATTR_CANDIDATES, props)
    spec = dict(footprint_layer=fp_layer, ground_attr=ground_attr,
                roof_attr=roof_attr, roof_lookup=None, layers=layers)
    if roof_attr is None:
        join_key = _first_present(JOIN_KEY_CANDIDATES, props)
        for rl in ROOFPART_LAYER_CANDIDATES:
            if rl not in layers or join_key is None:
                continue
            with fiona.open(src_path, layer=rl) as lyr:
                rprops = dict(lyr.schema["properties"])
            rattr = _first_present(ROOFPART_ROOF_CANDIDATES, rprops)
            if rattr is not None and join_key in rprops:
                spec["roof_lookup"] = dict(layer=rl, attr=rattr, key=join_key)
                spec["join_key"] = join_key
                break
    return spec

def _finite(v):
    try:
        return v is not None and math.isfinite(float(v))
    except (TypeError, ValueError):
        return False

def _build_roof_lookup(gpkg_path, spec):
    jl = spec["roof_lookup"]
    acc = {}
    with fiona.open(gpkg_path, layer=jl["layer"]) as lyr:
        for feat in lyr:
            pr = dict(feat["properties"])
            key, val = pr.get(jl["key"]), pr.get(jl["attr"])
            if key is None or not _finite(val):
                continue
            try:
                area = float(shape(feat["geometry"]).area)
            except Exception:
                area = 0.0
            wsum, warea, n = acc.get(key, (0.0, 0.0, 0))
            acc[key] = (wsum + float(val) * area, warea + area, n + 1)
    return {k: (ws / wa if wa > 0 else ws / max(n, 1)) for k, (ws, wa, n) in acc.items()}

def _open_gpkg(path):
    if path.endswith(".gpkg.gz"):
        out = path[:-len(".gz")]
        if not os.path.exists(out) or os.path.getsize(out) == 0:
            tmp = out + f".tmp{os.getpid()}"
            with gzip.open(path, "rb") as fi, open(tmp, "wb") as fo:
                shutil.copyfileobj(fi, fo)
            os.replace(tmp, out)
        return out
    if path.endswith(".zip"):
        return "/vsizip/" + os.path.abspath(path)
    return path

def _list_bag_sources():
    if bag_source == "local":
        return [bag_gpkg_path]
    return [os.path.join(BAG_TILES_DIR, fn) for fn in sorted(os.listdir(BAG_TILES_DIR)) if fn.endswith(".gpkg.gz")]

def build_index():
    hist = collections.Counter()
    n_feat = n_valid = n_invalid = 0
    out_schema = {"geometry": "MultiPolygon",
                  "properties": {"identificatie": "str", "h_rel": "float", "valid": "int"}}
    city_box = box(*aoi_bbox)
    sources = _list_bag_sources()
    assert sources, "no 3D BAG GeoPackage sources available"
    fgb_tmp = FGB_PATH + f".tmp{os.getpid()}"
    spec = None

    with fiona.open(fgb_tmp, "w", driver="FlatGeobuf", schema=out_schema, crs="EPSG:4326") as dst:
        for src_path in tqdm(sources, desc="3D BAG tiles", unit="tile"):
            path = _open_gpkg(src_path)
            if spec is None:
                spec = resolve_height_schema(path)
            roof_lookup = _build_roof_lookup(path, spec) if spec["roof_lookup"] is not None else None

            with fiona.open(path, layer=spec["footprint_layer"]) as src:
                tf = Transformer.from_crs(src.crs, "EPSG:4326", always_xy=True)
                reproject = lambda g: shp_transform(lambda x, y, z=None: tf.transform(x, y), g)

                for feat in src.filter(bbox=aoi_rd_bbox()):
                    g = feat.get("geometry")
                    if g is None:
                        continue
                    try:
                        geom = reproject(shape(g))
                    except Exception:
                        continue
                    if not geom.intersects(city_box):
                        continue
                    try:
                        clipped = geom.intersection(city_box)
                    except Exception:
                        try:
                            clipped = geom.buffer(0).intersection(city_box)
                        except Exception:
                            continue
                    if clipped.is_empty:
                        continue

                    pr = dict(feat["properties"])
                    roof = pr.get(spec["roof_attr"]) if spec["roof_attr"] else None
                    if roof_lookup is not None:
                        roof = roof_lookup.get(pr.get(spec["join_key"]))
                    ground = pr.get(spec["ground_attr"])
                    valid = int(_finite(roof) and _finite(ground))
                    h_rel = float(np.clip(float(roof) - float(ground), height_min, height_max)) if valid else 0.0

                    parts = [clipped] if clipped.geom_type == "Polygon" else list(getattr(clipped, "geoms", []))
                    for part in parts:
                        if part.geom_type != "Polygon" or part.is_empty:
                            continue
                        multi = {"type": "MultiPolygon", "coordinates": [part.__geo_interface__["coordinates"]]}
                        dst.write({"type": "Feature",
                                   "properties": {"identificatie": pr.get("identificatie") or "",
                                                  "h_rel": float(h_rel),
                                                  "valid": valid},
                                   "geometry": _FionaGeometry.from_dict(multi) if _FionaGeometry else multi})
                    n_feat += 1
                    if valid:
                        n_valid += 1
                        cent = clipped.representative_point()
                        tx, ty = lonlat_to_tile_f(cent.x, cent.y, zoom)
                        hist[(int(tx) // patch_tiles, int(ty) // patch_tiles)] += 1
                    else:
                        n_invalid += 1

    px = np.fromiter((k[0] for k in hist), dtype=np.int64, count=len(hist))
    py = np.fromiter((k[1] for k in hist), dtype=np.int64, count=len(hist))
    cnt = np.fromiter(hist.values(), dtype=np.int64, count=len(hist))
    os.replace(fgb_tmp, FGB_PATH)
    hist_tmp = HIST_PATH + f".tmp{os.getpid()}.npz"
    np.savez_compressed(hist_tmp, px=px, py=py, cnt=cnt)
    os.replace(hist_tmp, HIST_PATH)
    return px, py, cnt, dict(features=n_feat, valid=n_valid, invalid=n_invalid)

INDEX_READY = os.path.exists(FGB_PATH) and os.path.exists(HIST_PATH)
if resume and INDEX_READY:
    d = np.load(HIST_PATH)
    px_all, py_all, cnt_all = d["px"], d["py"], d["cnt"]
    set_stage("index_build", "complete")
else:
    px_all, py_all, cnt_all, counts = build_index()
    set_stage("index_build", "complete", **counts)

rng = np.random.default_rng(seed)
_keep = cnt_all >= min_buildings
px_all, py_all, cnt_all = px_all[_keep], py_all[_keep], cnt_all[_keep]

def save_split_files(splits, meta):
    for name, rows in splits.items():
        tmp = SPLIT_FILES[name] + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            for p, q, c in rows:
                f.write(f"{p} {q} {c}\n")
        os.replace(tmp, SPLIT_FILES[name])
    tmp = SPLIT_META + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    os.replace(tmp, SPLIT_META)

def load_split_files():
    out = {}
    for name, path in SPLIT_FILES.items():
        if not os.path.exists(path):
            return None
        rows = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                parts = line.split()
                if len(parts) >= 2:
                    rows.append((int(parts[0]), int(parts[1]), int(parts[2]) if len(parts) > 2 else 0))
        if not rows:
            return None
        out[name] = rows
    return out

def interleave_patches(splits, weights=(8, 1, 1)):
    names = ["train", "val", "test"]
    pools = {n: list(splits[n]) for n in names}
    pos = {n: 0 for n in names}
    pattern = [n for n, w in zip(names, weights) for _ in range(max(1, int(w)))]
    out, remaining = [], sum(len(p) for p in pools.values())
    while remaining > 0:
        moved = False
        for n in pattern:
            if pos[n] < len(pools[n]):
                p, q, _c = pools[n][pos[n]]
                pos[n] += 1
                out.append((n, p, q))
                remaining -= 1
                moved = True
        if not moved:
            break
    return out

auto_block = max(1, int(math.sqrt(max(1, len(px_all)) / 100.0)))
block_patches_eff = max(1, min(block_patches, auto_block))
block_x = (px_all // block_patches_eff).astype(np.int64)
block_y = (py_all // block_patches_eff).astype(np.int64)
blocks = sorted(set(zip(block_x.tolist(), block_y.tolist())), key=lambda b: morton(*b))
chunk_blocks = max(1, min(split_chunk_blocks, max(1, len(blocks) // 12)))
n_chunks = (len(blocks) + chunk_blocks - 1) // chunk_blocks

SPLIT_META_EXPECTED = dict(seed=seed, max_patches=max_patches,
                           block_patches=block_patches_eff,
                           split_chunk_blocks=chunk_blocks,
                           buffer_blocks=buffer_blocks,
                           region_patches=region_patches,
                           min_buildings=min_buildings,
                           train_frac=train_frac, val_frac=val_frac,
                           test_frac=test_frac, zoom=zoom,
                           patch_tiles=patch_tiles, aoi_name=aoi_name,
                           aoi_bbox=list(aoi_bbox), HIST_TAG=HIST_TAG)

loaded = load_split_files() if resume else None
meta_ok = False
buffer_eff = buffer_blocks

if loaded is not None and os.path.exists(SPLIT_META):
    try:
        _saved = json.load(open(SPLIT_META, encoding="utf-8"))
        meta_ok = all(_saved.get(k) == v for k, v in SPLIT_META_EXPECTED.items())
        buffer_eff = int(_saved.get("buffer_eff", buffer_blocks))
    except Exception:
        meta_ok = False

if loaded is not None and meta_ok:
    splits = loaded
    set_stage("split_generation", "complete")
else:
    block_chunk = {b: i // chunk_blocks for i, b in enumerate(blocks)}
    if n_chunks >= 10:
        def chunk_split(chunk):
            r = chunk % 10
            return "train" if r < 8 else ("val" if r == 8 else "test")
    else:
        chunk_names = []
        for chunk in range(n_chunks):
            frac = (chunk + 0.5) / n_chunks
            if frac < train_frac:
                chunk_names.append("train")
            elif frac < train_frac + val_frac:
                chunk_names.append("val")
            else:
                chunk_names.append("test")
        if "test" not in chunk_names:
            chunk_names[-1] = "test"
        if "val" not in chunk_names:
            idx = next((i for i in range(n_chunks - 1, -1, -1) if chunk_names[i] == "train"), None)
            assert idx is not None
            chunk_names[idx] = "val"
        def chunk_split(chunk):
            return chunk_names[chunk]

    block_split = {b: chunk_split(c) for b, c in block_chunk.items()}

    def conflicting(bx, by, split, buf):
        if split != "train":
            return False
        for dx in range(-buf, buf + 1):
            for dy in range(-buf, buf + 1):
                if block_split.get((bx + dx, by + dy)) in ("val", "test"):
                    return True
        return False

    def build_splits(buf):
        out = {"train": [], "val": [], "test": []}
        cand = {}
        for name, frac in (("train", train_frac), ("val", val_frac), ("test", test_frac)):
            idx = np.array([i for i, (bx, by) in enumerate(zip(block_x, block_y))
                            if block_split[(int(bx), int(by))] == name
                            and not conflicting(int(bx), int(by), name, buf)], dtype=np.int64)
            cand[name] = len(idx)
            if len(idx) == 0:
                continue
            chosen = idx if max_patches is None else idx[stratified_sample(px_all[idx], py_all[idx], cnt_all[idx], int(round(max_patches * frac)), rng)]
            for i in chosen:
                out[name].append((int(px_all[i]), int(py_all[i]), int(cnt_all[i])))
        return out, cand

    splits = None
    buffer_eff = buffer_blocks
    for buf_try in range(buffer_blocks, -1, -1):
        trial, cand = build_splits(buf_try)
        if all(len(trial[n]) > 0 for n in ("train", "val", "test")):
            splits, buffer_eff = trial, buf_try
            break

    def _blk(p, q):
        return (p // block_patches_eff, q // block_patches_eff)

    def _enforce_minimums(splits):
        vt = {_blk(p, q) for p, q, _ in splits["val"]} | {_blk(p, q) for p, q, _ in splits["test"]}
        for name in ("val", "test"):
            while len(splits[name]) < min_val_test and len(splits["train"]) > min_val_test:
                tr = [_blk(p, q) for p, q, _ in splits["train"]]
                if not tr:
                    break
                far = max(tr, key=lambda b: min((b[0] - w[0]) ** 2 + (b[1] - w[1]) ** 2 for w in vt) if vt else 0)
                for row in list(splits["train"]):
                    if _blk(row[0], row[1]) == far:
                        splits["train"].remove(row)
                        splits[name].append(row)
                        vt.add(far)
                        break
        return splits

    splits = _enforce_minimums(splits)
    save_split_files(splits, {**SPLIT_META_EXPECTED, "buffer_eff": buffer_eff})
    set_stage("split_generation", "complete")

for _n in ("train", "val", "test"):
    os.makedirs(os.path.join(IMG_DIR, _n), exist_ok=True)
    os.makedirs(os.path.join(TGT_DIR, _n), exist_ok=True)

PERMANENT_ERRORS = ("blank imagery", "no polygons", "no valid buildings", "no valid building pixels")

def rasterize_targets(feats, bbox, patch_x, patch_y):
    height = np.zeros((PATCH_PX, PATCH_PX), dtype=np.float32)
    mask = np.zeros((PATCH_PX, PATCH_PX), dtype=np.uint8)
    proj = PatchProjector(patch_x, patch_y)
    clip = box(*bbox)
    excl, valid_rings = [], []
    for geom, h_rel, valid in feats:
        try:
            clipped = geom.intersection(clip)
        except Exception:
            continue
        if clipped.is_empty:
            continue
        parts = [clipped] if clipped.geom_type == "Polygon" else list(getattr(clipped, "geoms", []))
        for part in parts:
            if part.geom_type != "Polygon" or part.is_empty:
                continue
            pts = proj.to_pixels(part)
            if len(pts) < 3:
                continue
            if valid:
                valid_rings.append((pts, float(h_rel)))
            else:
                excl.append(pts)
    if excl:
        cv2.fillPoly(mask, excl, 1)
    for pts, h_rel in valid_rings:
        cv2.fillPoly(mask, [pts], 1)
        cv2.fillPoly(height, [pts], float(np.clip(h_rel, height_min, height_max)))
    return height, mask

def process_patch(split, px_, py_, pid, img_path, tgt_path, src):
    tiles = patch_tile_list(px_, py_)
    try:
        img = stitch_patch(px_, py_)
        if img is None:
            raise RuntimeError("missing tiles")
        if np.asarray(img, dtype=np.float32).std() < blank_std_threshold:
            return dict(patch_id=pid, split=split, source=imagery_provider,
                        status="invalid", error="blank imagery"), "blank"
        bbox = patch_bbox_lonlat(px_, py_)
        feats = []
        for feat in src.filter(bbox=bbox):
            pr = dict(feat["properties"])
            try:
                feats.append((shape(feat["geometry"]), float(pr.get("h_rel") or 0.0), int(pr.get("valid") or 0)))
            except Exception:
                continue
        n_valid_bld = sum(f[2] for f in feats)
        if not feats:
            return dict(patch_id=pid, split=split, source=imagery_provider, status="invalid", error="no polygons"), "no_polygons"
        if n_valid_bld == 0:
            return dict(patch_id=pid, split=split, source=imagery_provider, status="invalid", error="no valid buildings"), "no_buildings"
        height, mask = rasterize_targets(feats, bbox, px_, py_)
        n_valid_px = int(mask.sum())
        if n_valid_px == 0:
            return dict(patch_id=pid, split=split, source=imagery_provider, status="invalid", error="no valid building pixels"), "no_buildings"

        img.save(img_path, quality=jpeg_quality)
        tmp = tgt_path + ".tmp.npz"
        np.savez_compressed(tmp, height=height, mask=mask)
        os.replace(tmp, tgt_path)
        return dict(patch_id=pid, split=split, source=imagery_provider,
                    status="complete", image_path=img_path, target_path=tgt_path,
                    file_size=os.path.getsize(img_path), n_polygons=len(feats),
                    n_valid_buildings=n_valid_bld, n_valid_px=n_valid_px,
                    validated=True, error=""), "downloaded"
    except Exception as e:
        return dict(patch_id=pid, split=split, source=imagery_provider, status="failed", error=str(e)[:200]), "failed"
    finally:
        drop_tiles(tiles)

all_patches = interleave_patches(splits, (round(train_frac * 10), round(val_frac * 10), round(test_frac * 10)))
if build_limit is not None:
    all_patches = all_patches[:build_limit]

manifest = load_manifest() if resume else {}
stats = collections.Counter()
set_stage("patch_generation", "in_progress")

with cf.ThreadPoolExecutor(max_workers=download_workers) as ex, fiona.open(FGB_PATH) as src:
    for start in range(0, len(all_patches), patch_batch):
        patch_chunk = all_patches[start:start + patch_batch]
        todo = []
        for name, p, q in patch_chunk:
            pid = f"{p}_{q}"
            img_path = os.path.join(IMG_DIR, name, pid + ".jpg")
            tgt_path = os.path.join(TGT_DIR, name, pid + ".npz")
            rec = manifest.get(pid)
            if (resume and rec and rec.get("status") == "complete" and rec.get("split") == name
                    and valid_image(img_path) and valid_target(tgt_path)):
                stats["already_existing"] += 1
                continue
            if (resume and rec and rec.get("status") == "invalid" and rec.get("error") in PERMANENT_ERRORS
                    and rec.get("split") == name):
                stats["skipped"] += 1
                continue
            todo.append((name, p, q, pid, img_path, tgt_path))

        tiles = [t for (_n, p, q, *_rest) in todo for t in patch_tile_list(p, q)]
        if tiles:
            list(ex.map(lambda t: fetch_tile(*t), tiles))

        new_records = []
        for name, p, q, pid, img_path, tgt_path in todo:
            rec, kind = process_patch(name, p, q, pid, img_path, tgt_path, src)
            stats[kind] += 1
            manifest[pid] = rec
            new_records.append(rec)
        append_manifest(new_records)

built = {n: 0 for n in ("train", "val", "test")}
for r in manifest.values():
    if r.get("status") == "complete" and r.get("split") in built:
        built[r["split"]] += 1
complete = sum(1 for r in manifest.values() if r.get("status") == "complete")
set_stage("patch_generation", "complete" if stats["failed"] == 0 else "in_progress",
          failed=stats["failed"], complete=complete, **built)

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def split_entries(name):
    rows = []
    for p, q, _c in splits[name]:
        pid = f"{p}_{q}"
        ip = os.path.join(IMG_DIR, name, pid + ".jpg")
        tp = os.path.join(TGT_DIR, name, pid + ".npz")
        if os.path.exists(ip) and valid_target(tp):
            rows.append((pid, ip, tp))
    return rows

class BagHeightDataset(torch.utils.data.Dataset):
    def __init__(self, split, imgsz=model_imgsz):
        self.split = split
        self.imgsz = imgsz
        self.entries = split_entries(split)
        assert self.entries, f"no complete patches for split '{split}'"

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, i):
        pid, ip, tp = self.entries[i]
        bgr = cv2.imread(ip, cv2.IMREAD_COLOR)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        if rgb.shape[0] != self.imgsz or rgb.shape[1] != self.imgsz:
            rgb = cv2.resize(rgb, (self.imgsz, self.imgsz), interpolation=cv2.INTER_LINEAR)
        x = rgb.astype(np.float32) / 255.0
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        x = np.ascontiguousarray(x.transpose(2, 0, 1))

        with np.load(tp) as d:
            h, m = d["height"], d["mask"]
        if h.shape[0] != self.imgsz or h.shape[1] != self.imgsz:
            h = cv2.resize(h, (self.imgsz, self.imgsz), interpolation=cv2.INTER_AREA)
            m = cv2.resize(m, (self.imgsz, self.imgsz), interpolation=cv2.INTER_NEAREST)
        h = np.clip(h.astype(np.float32), height_min, height_max)
        m = (m > 0).astype(np.float32)
        return dict(x=torch.from_numpy(x), y=torch.from_numpy(h[None, ...]), m=torch.from_numpy(m), pid=pid)

def make_loaders():
    g = torch.Generator()
    g.manual_seed(seed)
    train_ds = BagHeightDataset("train")
    val_ds = BagHeightDataset("val")
    test_ds = BagHeightDataset("test")
    train_dl = torch.utils.data.DataLoader(
        train_ds, batch_size=micro_batch, shuffle=True, drop_last=False,
        num_workers=num_workers, pin_memory=(device != "cpu"), persistent_workers=(num_workers > 0), generator=g)
    val_dl = torch.utils.data.DataLoader(
        val_ds, batch_size=micro_batch, shuffle=False, num_workers=num_workers, pin_memory=(device != "cpu"))
    test_dl = torch.utils.data.DataLoader(
        test_ds, batch_size=micro_batch, shuffle=False, num_workers=num_workers, pin_memory=(device != "cpu"))
    return train_dl, val_dl, test_dl

train_dl, val_dl, test_dl = make_loaders()

AMP_ON = bool(amp) and device != "cpu"

def make_grad_scaler(enabled=True):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)

import contextlib
@contextlib.contextmanager
def amp_autocast(enabled=False):
    if not enabled:
        yield
        return
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        with torch.amp.autocast(device_type="cuda", enabled=True):
            yield
    else:
        with torch.cuda.amp.autocast(enabled=True):
            yield

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class DecoderBlock(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv = ConvBlock(cin + cskip, cout)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class ResNet18UNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        enc = resnet18(weights=_IMAGENET1K if pretrained else None)
        self.stem = nn.Sequential(enc.conv1, enc.bn1, enc.relu)
        self.pool = enc.maxpool
        self.layer1 = enc.layer1
        self.layer2 = enc.layer2
        self.layer3 = enc.layer3
        self.layer4 = enc.layer4
        self.up1 = DecoderBlock(512, 256, 256)
        self.up2 = DecoderBlock(256, 128, 128)
        self.up3 = DecoderBlock(128, 64, 64)
        self.up4 = DecoderBlock(64, 64, 64)
        self.head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1),
        )
        self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)

    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.layer1(self.pool(s0))
        s2 = self.layer2(s1)
        s3 = self.layer3(s2)
        s4 = self.layer4(s3)
        d = self.up1(s4, s3)
        d = self.up2(d, s2)
        d = self.up3(d, s1)
        d = self.up4(d, s0)
        return self.upsample(self.head(d))

def masked_height_loss(pred, target, mask, lambda_bg_w=None, beta=None):
    lambda_bg_w = lambda_bg if lambda_bg_w is None else lambda_bg_w
    beta = huber_beta if beta is None else beta
    p, t = pred[:, 0], target[:, 0]
    bmask = mask > 0.5
    bgmask = ~bmask
    loss_b = F.smooth_l1_loss(p[bmask], t[bmask], reduction="mean", beta=beta) if bmask.any() else p.sum() * 0.0
    loss_bg = F.smooth_l1_loss(p[bgmask], t[bgmask], reduction="mean", beta=beta) if bgmask.any() else p.sum() * 0.0
    return loss_b + lambda_bg_w * loss_bg, loss_b, loss_bg

class PixelAccumulator:
    def __init__(self):
        self.n = 0
        self.s_abs = 0.0
        self.s_sq = 0.0
        self.s_x = self.s_y = self.s_xx = self.s_yy = self.s_xy = 0.0

    def add(self, pred, target, mask):
        sel = mask.reshape(-1) > 0.5
        if not np.any(sel):
            return
        p = pred.reshape(-1)[sel].astype(np.float64)
        t = target.reshape(-1)[sel].astype(np.float64)
        d = p - t
        self.n += p.size
        self.s_abs += float(np.abs(d).sum())
        self.s_sq += float((d * d).sum())
        self.s_x += float(p.sum())
        self.s_y += float(t.sum())
        self.s_xx += float((p * p).sum())
        self.s_yy += float((t * t).sum())
        self.s_xy += float((p * t).sum())

    def results(self):
        if self.n == 0:
            return dict(mae=None, rmse=None, r=None, n_pixels=0)
        mae = self.s_abs / self.n
        rmse = math.sqrt(self.s_sq / self.n)
        r = None
        if self.n >= 10:
            cov = self.n * self.s_xy - self.s_x * self.s_y
            vx = self.n * self.s_xx - self.s_x ** 2
            vy = self.n * self.s_yy - self.s_y ** 2
            if vx > 0 and vy > 0:
                r = float(cov / math.sqrt(vx * vy))
        return dict(mae=float(mae), rmse=float(rmse), r=r, n_pixels=int(self.n))

@torch.no_grad()
def evaluate(model, dl, per_patch=False):
    model.eval()
    acc = PixelAccumulator()
    rows = []
    for batch in dl:
        x = batch["x"].to(device, non_blocking=True)
        y = batch["y"].numpy()
        m = batch["m"].numpy()
        with amp_autocast(AMP_ON):
            pred = model(x)
        pred = pred.float().cpu().numpy()
        acc.add(pred[:, 0], y[:, 0], m)
        if per_patch:
            for i, pid in enumerate(batch["pid"]):
                pa = PixelAccumulator()
                pa.add(pred[i, 0], y[i, 0], m[i])
                rows.append(dict(patch_id=pid, **pa.results()))
    out = acc.results()
    if per_patch:
        out["per_patch"] = rows
    return out

def is_cuda_oom(exc):
    oom_type = getattr(torch.cuda, "OutOfMemoryError", None)
    if oom_type is not None and isinstance(exc, oom_type):
        return True
    return "out of memory" in str(exc).lower()

def free_gpu_memory():
    import gc
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass

def micro_batch_candidates(start):
    out, b = [], max(1, int(start))
    while True:
        if b not in out and effective_batch % b == 0:
            out.append(b)
        if b <= 1:
            break
        b = max(1, b // 2)
    if 1 not in out:
        out.append(1)
    return out

def atomic_torch_save(obj, path):
    tmp = f"{path}.tmp{os.getpid()}"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def torch_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def _stack_batch(dataset, idxs):
    items = [dataset[int(j)] for j in idxs]
    x = torch.stack([it["x"] for it in items]).to(device, non_blocking=True)
    y = torch.stack([it["y"] for it in items]).to(device, non_blocking=True)
    m = torch.stack([it["m"] for it in items]).to(device, non_blocking=True)
    return x, y, m

def train_epoch(model, dataset, optimizer, scaler, epoch, micro_bs):
    model.train()
    n = len(dataset)
    g = torch.Generator()
    g.manual_seed(seed * 100003 + epoch)
    perm = torch.randperm(n, generator=g).tolist()
    micro = int(micro_bs)
    cursor = 0
    optimizer.zero_grad(set_to_none=True)
    agg = collections.Counter()
    while cursor < n:
        window_start = cursor
        accum = max(1, effective_batch // micro)
        window = perm[window_start:window_start + accum * micro]
        n_chunks = max(1, math.ceil(len(window) / micro))
        try:
            pos = 0
            while pos < len(window):
                idx = window[pos:pos + micro]
                pos += len(idx)
                x, y, m = _stack_batch(dataset, idx)
                with amp_autocast(AMP_ON):
                    pred = model(x)
                    loss, l_b, l_bg = masked_height_loss(pred, y, m)
                scaler.scale(loss / n_chunks).backward()
                agg["loss"] += float(loss.detach()) / n_chunks
                agg["bld"] += float(l_b.detach())
                agg["bg"] += float(l_bg.detach())
                agg["micro_batches"] += 1
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            cursor = window_start + len(window)
            agg["windows"] += 1
            record_batch_state(good=micro)
        except Exception as e:
            if not is_cuda_oom(e):
                raise
            record_batch_state(failed=micro)
            free_gpu_memory()
            smaller = [b for b in LADDER if b < micro]
            optimizer.zero_grad(set_to_none=True)
            if not smaller:
                raise RuntimeError(f"CUDA OOM at micro_batch={micro}") from e
            micro = smaller[0]
            cursor = window_start
    n_w = max(1, agg["windows"])
    return dict(loss=agg["loss"] / n_w, bld=agg["bld"] / n_w, bg=agg["bg"] / n_w,
                windows=agg["windows"], micro=micro), micro

model = ResNet18UNet(pretrained=True).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1), eta_min=lr * 0.01)
scaler = make_grad_scaler(enabled=AMP_ON)
LADDER = micro_batch_candidates(micro_batch)
cur_micro = LADDER[0]

start_epoch = 0
best_val_rmse = float("inf")
best_epoch = -1

if resume and os.path.exists(LAST_PT):
    ck = torch_load(LAST_PT)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    start_epoch = int(ck.get("epoch", -1)) + 1
    best_val_rmse = float(ck.get("best_val_rmse", float("inf")))

if resume and os.path.exists(BEST_PT):
    bck = torch_load(BEST_PT)
    if float(bck.get("best_val_rmse", float("inf"))) <= best_val_rmse:
        best_val_rmse = float(bck.get("best_val_rmse", float("inf")))
        best_epoch = int(bck.get("epoch", -1))

def _ckpt_payload(epoch):
    return dict(epoch=epoch, model=model.state_dict(), optimizer=optimizer.state_dict(),
                scaler=scaler.state_dict() if AMP_ON else None, scheduler=scheduler.state_dict(),
                best_val_rmse=best_val_rmse, config=CONFIG, model_imgsz=model_imgsz,
                height_min=height_min, height_max=height_max)

if start_epoch < epochs:
    set_stage("training", "in_progress", epoch=start_epoch, total=epochs)
    for epoch in range(start_epoch, epochs):
        tr_stats, used_micro = train_epoch(model, train_dl.dataset, optimizer, scaler, epoch, cur_micro)
        cur_micro = used_micro
        val = evaluate(model, val_dl)
        rmse = val["rmse"] if val["rmse"] is not None else float("nan")
        improved = math.isfinite(rmse) and rmse < best_val_rmse
        if improved:
            best_val_rmse = rmse
            best_epoch = epoch
            atomic_torch_save({**_ckpt_payload(epoch), "val": val}, BEST_PT)
        scheduler.step()
        atomic_torch_save(_ckpt_payload(epoch), LAST_PT)
    set_stage("training", "complete", epoch=epochs, total=epochs, best_val_rmse=best_val_rmse, best_epoch=best_epoch)

if os.path.exists(BEST_PT):
    model = ResNet18UNet(pretrained=False).to(device)
    _best_ckpt = torch_load(BEST_PT)
    model.load_state_dict(_best_ckpt["model"])
    model.eval()

    metrics = dict(
        checkpoint=BEST_PT,
        checkpoint_epoch=int(_best_ckpt.get("epoch", -1)) + 1,
        best_val_rmse_m=round(float(_best_ckpt.get("best_val_rmse", float("nan"))), 4),
        model_imgsz=model_imgsz,
        zoom=zoom,
        gsd_m_per_px=round(GSD_M_PER_PX, 4),
        patch_px=PATCH_PX,
        height_clamp_m=[height_min, height_max],
        loss=dict(kind="SmoothL1", beta=huber_beta, lambda_bg=lambda_bg),
        scope="valid building pixels only (building_mask == 1)",
    )
    for _split in ("val", "test"):
        _dl = val_dl if _split == "val" else test_dl
        _r = evaluate(model, _dl, per_patch=True)
        _rows = sorted([row for row in _r.pop("per_patch") if row["mae"] is not None], key=lambda row: -row["mae"])
        _r["n_patches"] = len(_dl.dataset)
        metrics[_split] = _r
        metrics[_split]["worst_patches"] = _rows[:5]

    _tmp = METRICS_PATH + ".tmp"
    with open(_tmp, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    os.replace(_tmp, METRICS_PATH)
    set_stage("evaluation", "complete", val_rmse=metrics["val"]["rmse"], test_rmse=metrics["test"]["rmse"])

# Building-Level (Per-Instance) Evaluation & Complete GeoJSON Output
ZOOM_HI = 19
HI_TILES = 5
HI_PX = HI_TILES * 256
MIN_PX_AREA = 25.0

_TF_4326_28992 = Transformer.from_crs("EPSG:4326", "EPSG:28992", always_xy=True)
_res19 = webmercator_resolution(ZOOM_HI)

def _poly_area(pts):
    x = pts[:, 0].astype(np.float64)
    y = pts[:, 1].astype(np.float64)
    return 0.5 * abs(float(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))))

def _fetch_hi(z, x, y):
    fp = os.path.join(CACHE_DIR, f"{z}_{x}_{y}.jpg")
    if os.path.exists(fp) and os.path.getsize(fp) > 0:
        return fp
    try:
        r = _session().get(TILE_URL.format(z=z, y=y, x=x), timeout=20)
        if r.status_code == 200 and r.content and len(r.content) > 100:
            with open(fp, "wb") as f:
                f.write(r.content)
            return fp
    except Exception:
        pass
    return None

def _stitch_hi(ox, oy):
    canvas = Image.new("RGB", (HI_PX, HI_PX), (0, 0, 0))
    ok = 0
    for dy in range(HI_TILES):
        for dx in range(HI_TILES):
            fp = os.path.join(CACHE_DIR, f"{ZOOM_HI}_{ox + dx}_{oy + dy}.jpg")
            if not os.path.exists(fp):
                continue
            canvas.paste(Image.open(fp).convert("RGB"), (dx * 256, dy * 256))
            ok += 1
    return canvas if ok == HI_TILES * HI_TILES else None

def _to_hi_pts(g3857, origin_x19, origin_y19):
    xy = np.asarray(g3857.exterior.coords, dtype=np.float64)
    px = (xy[:, 0] - origin_x19) / _res19
    py = (origin_y19 - xy[:, 1]) / _res19
    return np.stack([px, py], axis=1)

all_records = []
all_evaluated_buildings = []  # Aggregate across all test patches
test_rows = sorted(split_entries("test"), key=lambda e: e[0])

for pid, ip, tp in test_rows:
    px_, py_ = int(pid.split("_")[0]), int(pid.split("_")[1])
    bbox = patch_bbox_lonlat(px_, py_)
    proj16 = PatchProjector(px_, py_)

    bgr = cv2.imread(ip, cv2.IMREAD_COLOR)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    rgb640 = cv2.resize(rgb, (model_imgsz, model_imgsz), interpolation=cv2.INTER_LINEAR)
    xin = (rgb640.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
    xin = np.ascontiguousarray(xin.transpose(2, 0, 1))
    with torch.no_grad():
        pred = model(torch.from_numpy(xin)[None].to(device))
    pred_h = np.clip(pred[0, 0].float().cpu().numpy(), height_min, height_max)

    with np.load(tp) as d:
        gt_h = d["height"]
    if gt_h.shape[0] != model_imgsz:
        gt_h = cv2.resize(gt_h, (model_imgsz, model_imgsz), interpolation=cv2.INTER_AREA)
    gt_h = np.clip(gt_h.astype(np.float32), height_min, height_max)

    buildings = []
    with fiona.open(FGB_PATH) as src:
        for feat in src.filter(bbox=bbox):
            pr = dict(feat["properties"])
            if int(pr.get("valid") or 0) != 1 or feat.get("geometry") is None:
                continue
            try:
                geom = shape(feat["geometry"])
            except Exception:
                continue
            if geom.geom_type == "MultiPolygon":
                geom = max(geom.geoms, key=lambda p: p.area)
            if geom.geom_type != "Polygon" or len(geom.exterior.coords) < 3:
                continue
            g3857 = shp_transform(lambda a, b, z=None: _TF_4326_3857.transform(a, b), geom)
            g28992 = shp_transform(lambda a, b, z=None: _TF_4326_28992.transform(a, b), geom)
            pts = proj16.to_pixels(geom)
            if len(pts) < 3 or _poly_area(pts) < MIN_PX_AREA:
                continue
            rp = g3857.representative_point()
            buildings.append(dict(geom=geom, g3857=g3857, pts=pts, px_area=_poly_area(pts),
                                  area_m2=float(g28992.area), rp=(rp.x, rp.y),
                                  identificatie=str(pr.get("identificatie") or "")))

    for i, b in enumerate(buildings):
        b["id"] = f"{pid}_{i}"
        m = np.zeros((model_imgsz, model_imgsz), dtype=np.uint8)
        cv2.fillPoly(m, [b["pts"].reshape(-1, 1, 2)], 1)
        sel = m > 0
        if not sel.any():
            continue
        gt_val = float(np.percentile(gt_h[sel], 70))
        pred_val = float(np.percentile(pred_h[sel], 70))
        abs_err = abs(pred_val - gt_val)
        rel_err = abs_err / max(gt_val, 2.0) * 100.0
        b.update(gt=gt_val, pred=pred_val, abs_err=abs_err, rel_err=rel_err)
        all_records.append((pid, b["id"], b["identificatie"], round(b["area_m2"], 2),
                            round(gt_val, 3), round(pred_val, 3), round(abs_err, 3), round(rel_err, 2)))

    all_evaluated_buildings.extend(buildings)

# Export Full CSV
tmp = BL_CSV + ".tmp"
with open(tmp, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["patch_id", "building_id", "identificatie", "area_m2",
                "gt_height_p70_m", "pred_height_p70_m", "abs_error_m", "rel_error_pct"])
    for rec in all_records:
        w.writerow(rec)
os.replace(tmp, BL_CSV)

# Export Full Multi-Patch GeoJSON
geojson_features = []
for b in all_evaluated_buildings:
    geom = b.get("geom")
    gt = b.get("gt")
    pred = b.get("pred")
    if geom is None or gt is None or pred is None:
        continue

    err_signed = pred - gt
    abs_err = b.get("abs_err", abs(err_signed))
    rel_err = b.get("rel_err", (abs_err / max(gt, 1e-3)) * 100.0)

    feature = {
        "type": "Feature",
        "geometry": mapping(geom),
        "properties": {
            "building_id": b.get("id", ""),
            "identificatie": b.get("identificatie", ""),
            "area_m2": round(float(b.get("area_m2", 0.0)), 1),
            "height_gt": round(float(gt), 2),
            "height_pred": round(float(pred), 2),
            "error_abs": round(float(abs_err), 2),
            "error_signed": round(float(err_signed), 2),
            "error_pct": round(float(rel_err), 1),
        }
    }
    geojson_features.append(feature)

out_geojson_path = os.path.join(BL_DIR, "amsterdam_3d_buildings.geojson")
with open(out_geojson_path, "w", encoding="utf-8") as f:
    json.dump({"type": "FeatureCollection", "features": geojson_features}, f, ensure_ascii=False)

log(f"[BUILDING-EVAL] Complete GeoJSON saved ({len(geojson_features):,} buildings) -> {out_geojson_path}")